# Analyzing some tendencies after slim dimensionality reduction with Random Embeddings

In this script we just focus on some partial dataset wherein we use $d=20$ on just a subset of functions from BBOB (i.e. $f=\{1,8,11,16,20 \}$). What we want to do is then check how some features from the Exploratory Landscape Analysis Toolbox (i.e. FLACCO) behave under a projection with Random Embeddings with a projection matrix $P \in \mathbb{R}^{d \times q}$. In this case we set $q=10$ as a preliminary value to have $50\%$ reduction.

## Procedure
For this case, we use the package `sklearn.random_projection.GaussianRandomProjection` in order to generate the random projection matrices. This generates random matrices where $\left\|P \right\|_{2}=1.0$. With this in mind, we generate 10 different random projection matrices of this kind. Furthermore, we generated beforehand initial 10 datasets of $\mathbf{X} \in [-5,5]^{d}$ with $100 d$ samples via `LHS`. *A priori*, the sampling scheme wouldn't signify a very different performance with respect to other sampling available techniques since the Random Projection will transform any initial space-filling optimized Design of Experiments scheme into some degenerate one since there'll be artificial correlations caused by the projection into the reduced latent space.

### Sampled ELA Features
After what was discussed in some references (i.e. *Expressiveness and Robustness of Landscape Features*, Renau et al., 2019), we sampled the following sets of ELA Features:
- **Dispersion (disp)**
- **Information Content (ic)**
- **Nearest Best Clustering (nbc)**
- **Principal Component Analysis (pca)**
- **Meta Model (ela_meta)**
- **$y$-distribution (ela_distr)**
- **Level Set (ela_level)**

From the set of features, it's self-evident that the **$y$-distribution** features, these are invariant to any dimensionality reduction techniques since the scope of these features is just the *objective space*. However to account for subsampling in a high dimensional space we proceed to generate subsets of $100 q$ by sampling without replacement from the initial dataset. The idea is then two-fold: 1) We simulate what would happen what happens by keeping the same *dimensionality propertion* and 2) we can study the stability and possible distributions of the features. We hypothesize that the distributions of features by performing this sampling will depend primarily on the sampled function.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# Import necessary libraries
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# Define file paths
ela_features_reduced_path = Path("ela_features_reduced_3/ELA_extraction/D_20").resolve()
ela_features_complete_path = Path("ela_features_3/ELA_extraction/Dimension_20").resolve()

In [ ]:
# Get list of CSV files in both directories
reduced_files = sorted(ela_features_reduced_path.rglob("*.csv"))
complete_files = sorted(ela_features_complete_path.rglob("*.csv"))

In [ ]:
complete_files

In [ ]:
def extract_meta_data_from_reduced_feature_file_path(file_path: Path) -> dict:
    """
    Extracts meta data from the reduced feature file path.
    
    Parameters:
    file_path (Path): Path object of the reduced feature file.
    
    Returns:
    dict: A dictionary containing the extracted meta data.
    """
    """
    Extract key-value numeric metadata from path segments like key_value.
    """

    if not isinstance(file_path, (str, Path)):
        raise ValueError("path must be a string or Path object")

    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"The path {file_path} does not exist.")

    metadata = {}

    round_txt = file_path.parts[-1]
    round_number = int(round_txt.split("_")[-1].replace(".csv", ""))
    metadata["round"] = round_number

    embedding_seed_txt = file_path.parts[-2]
    embedding_seed = int(embedding_seed_txt.split("_")[-1])
    metadata["embedding_seed"] = embedding_seed

    reduction_ratio_txt = file_path.parts[-4]
    reduction_ratio = float(reduction_ratio_txt.split("_")[-1])
    metadata["reduction_ratio"] = reduction_ratio

    instance_idx_txt = file_path.parts[-5]
    instance_idx = int(instance_idx_txt.split("_")[-1])
    metadata["instance_idx"] = instance_idx

    function_idx_txt = file_path.parts[-6]
    function_idx = int(function_idx_txt.split("_")[-1])
    metadata["function_idx"] = function_idx

    n_samples_txt = file_path.parts[-7]
    n_samples = int(n_samples_txt.split("_")[-1])
    metadata["n_samples"] = n_samples

    seed_txt = file_path.parts[-8]
    seed = int(seed_txt.split("_")[-1])
    metadata["seed_lhs"] = seed

    dimension_txt = file_path.parts[-9]
    dimension = int(dimension_txt.split("_")[-1])
    metadata["dimension"] = dimension
    
    return metadata

In [ ]:
def extract_meta_data_from_complete_feature_file_path(file_path: Path) -> dict:
    """
    Extracts meta data from the complete feature file path.
    
    Parameters:
    file_path (Path): Path object of the complete feature file.
    
    Returns:
    dict: A dictionary containing the extracted meta data.
    """
    """
    Extract key-value numeric metadata from path segments like key_value.
    """

    if not isinstance(file_path, (str, Path)):
        raise ValueError("path must be a string or Path object")

    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"The path {file_path} does not exist.")

    metadata = {}



    instance_idx_txt = file_path.parts[-2]
    instance_idx = int(instance_idx_txt.split("_")[-1])
    metadata["instance_idx"] = instance_idx

    function_idx_txt = file_path.parts[-3]
    function_idx = int(function_idx_txt.split("_")[-1])
    metadata["function_idx"] = function_idx

    n_samples_txt = file_path.parts[-4]
    n_samples = int(n_samples_txt.split("_")[-1])
    metadata["n_samples"] = n_samples

    seed_txt = file_path.parts[-5]
    seed = int(seed_txt.split("_")[-1])
    metadata["seed_lhs"] = seed

    dimension_txt = file_path.parts[-6]
    dimension = int(dimension_txt.split("_")[-1])
    metadata["dimension"] = dimension
    
    return metadata

In [ ]:
# Generate DataFrames by loading CSV files and combining with metadata
complete_dfs = []

for file_path in complete_files:
    df = pd.read_csv(file_path)
    # Extract metadata for the current file
    meta = extract_meta_data_from_complete_feature_file_path(file_path)
    # Add metadata as new columns to the DataFrame
    for key, value in meta.items():
        df[key] = value
    
    complete_dfs.append(df)

complete_df = (
    pd.concat(complete_dfs, ignore_index=True)
    if complete_dfs
    else pd.DataFrame()
)

In [ ]:
complete_df

In [ ]:
complete_df

In [ ]:
FEATURE_DTYPES = {
    col: "float64"
    for col in pd.read_csv(
        reduced_files[0],
        nrows=0
    ).columns
}

META_DTYPES = {
    "dimension": "int16",
    "seed_lhs": "int32",
    "n_samples": "int32",
    "function_idx": "int16",
    "instance_idx": "int32",
    "reduction_ratio": "float32",
    "embedding_seed": "int32",
    "round": "int16",
}

In [ ]:
# Generate DataFrames by loading CSV files and combining with metadata
reduced_dfs = []

for file_path in reduced_files:
    df = pd.read_csv(file_path)
    # Extract metadata for the current file
    meta = extract_meta_data_from_reduced_feature_file_path(file_path)
    # Add metadata as new columns to the DataFrame
    for key, value in meta.items():
        df[key] = value
    
    reduced_dfs.append(df)

reduced_df = (
    pd.concat(reduced_dfs, ignore_index=True)
    if reduced_dfs
    else pd.DataFrame()
)

In [ ]:
len(reduced_files)

In [ ]:
def load_reduced(fp):
    # Fast CSV read, no inference
    df = pd.read_csv(
        fp,
        dtype=FEATURE_DTYPES,
        low_memory=False,
        engine="c"
    )

    meta = extract_meta_data_from_reduced_feature_file_path(fp)

    # Typed metadata injection (no object columns)
    return df.assign(
        **{
            k: pd.Series(v, dtype=META_DTYPES[k], index=df.index)
            for k, v in meta.items()
        }
    )



In [ ]:
def load_concat_chunk(files):
    return pd.concat(
        map(load_reduced, files),
        ignore_index=True,
        copy=False
    )

In [ ]:
chunks = []

CHUNK_SIZE = 10_000

for i in range(0, len(reduced_files), CHUNK_SIZE):
    chunk_files = reduced_files[i:i + CHUNK_SIZE]
    chunk_df = load_concat_chunk(chunk_files)
    chunks.append(chunk_df)

reduced_df = pd.concat(chunks, ignore_index=True, copy=False)

In [ ]:
# 3. Concatenate everything
reduced_df = pd.concat(
    map(load_reduced, reduced_files),
    ignore_index=True
)

In [ ]:
# Store the final DataFrame
reduced_df.to_parquet("reduced_data_2.parquet", engine="pyarrow", compression="snappy")


In [ ]:
complete_df.to_csv("complete_data_2.csv", index=False)

In [ ]:
reduced_df.to_csv("reduced_data_2.csv", index=False)